In [33]:
# ═══════════════════════════════════════════════════════════════════════
# CYBERPATH v2 — AI-POWERED RECOMMENDATION SYSTEM

# This notebook builds an independent, supervised machine learning
# recommendation system trained on synthetic student-machine relevance data.
#
# It is SEPARATE from the v1 system (recommender.ipynb), which remains
# unchanged and continues to use dataset_final.csv.
#
# v2 uses the corrected dataset (dataset_v2_ai_clean.csv) and trains two
# supervised ranking models for comparison: Random Forest and XGBoost.

# Import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Set up paths — relative to the notebook location
BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'data', 'dataset_v2_ai_clean.csv')
MODELS_DIR_V2 = os.path.join(BASE_DIR, 'saved_models', 'ai_recommender')

# Create the v2 models folder if it doesn't exist (won't touch v1's folder)
os.makedirs(MODELS_DIR_V2, exist_ok=True)
os.makedirs(os.path.join(MODELS_DIR_V2, 'plots'), exist_ok=True)

print(f"BASE_DIR:       {BASE_DIR}")
print(f"DATA_PATH:      {DATA_PATH}")
print(f"MODELS_DIR_V2:  {MODELS_DIR_V2}")
print(f"\nDataset file exists: {os.path.exists(DATA_PATH)}")
print(f"v2 folder created:   {os.path.exists(MODELS_DIR_V2)}")

BASE_DIR:       c:\curtin\project new\Vuln Recommendation system
DATA_PATH:      c:\curtin\project new\Vuln Recommendation system\data\dataset_v2_ai_clean.csv
MODELS_DIR_V2:  c:\curtin\project new\Vuln Recommendation system\saved_models\ai_recommender

Dataset file exists: True
v2 folder created:   True


In [34]:
# load dataset and check basics

df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print("columns:", list(df.columns))

# any missing values?
print("\nmissing:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# check distributions of main fields
print("\ndifficulty:")
print(df['Difficulty'].value_counts())

print("\nattack category:")
print(df['Attack_Category'].value_counts())

print("\nos:")
print(df['OS'].value_counts())

print("\nestimated time:")
print(df['Estimated_Time'].value_counts())

# how many unique learning objectives
all_objs = []
for s in df['Learning_Objectives'].dropna():
    all_objs.extend([x.strip() for x in str(s).split(';') if x.strip()])
print("\nunique learning objectives:", len(set(all_objs)))

# quick look at first 3 rows
df[['Machine_Name', 'Difficulty', 'Attack_Category', 'OS', 'Estimated_Time']].head(3)

shape: (305, 18)
columns: ['Machine_ID', 'Machine_Name', 'Platform', 'Platform_ID', 'OS', 'OS_Detail', 'Difficulty', 'Difficulty_Numeric', 'Attack_Category', 'Attack_Category_Detail', 'Vulnerability_Type', 'Kill_Chain_Stages', 'Skills_Required', 'Learning_Objectives', 'Estimated_Time', 'Estimated_Time_Hours', 'Attack_Path_Length', 'Entry_Point']

missing:
OS_Detail    67
dtype: int64

difficulty:
Difficulty
Medium    159
Hard      106
Easy       40
Name: count, dtype: int64

attack category:
Attack_Category
Web Exploitation              190
Network Exploitation          107
Binary Exploitation             4
Mixed (Web + Network)           3
Cryptographic Exploitation      1
Name: count, dtype: int64

os:
OS
Linux      303
FreeBSD      1
Windows      1
Name: count, dtype: int64

estimated time:
Estimated_Time
2-3 hours    140
1-2 hours    103
3-4 hours     43
4+ hours      11
<1 hour        8
Name: count, dtype: int64

unique learning objectives: 99


,Machine_Name,Difficulty,Attack_Category,OS,Estimated_Time
0,LAMPSecurity CTF 4,Easy,Web Exploitation,Linux,1-2 hours
1,LAMPSecurity CTF 5,Easy,Web Exploitation,Linux,2-3 hours
2,LAMPSecurity CTF 7,Easy,Web Exploitation,Linux,2-3 hours


In [35]:
sample = str(df['Kill_Chain_Stages'].iloc[0])
print("sample:", sample)
print("contains arrow:", '→' in sample)

sample: Recon → Exploitation → Credential Access → Lateral Movement → Privilege Escalation
contains arrow: True


In [36]:
# encode machines into numeric features for the ml model

from sklearn.preprocessing import MultiLabelBinarizer

# difficulty as number
diff_map = {'Easy': 1, 'Medium': 2, 'Hard': 3}
df['diff_num'] = df['Difficulty'].map(diff_map)

# count kill chain stages per machine
def count_chain_stages(s):
    if pd.isna(s): return 0
    return len([p for p in str(s).split('→') if p.strip()])

df['kill_chain_count'] = df['Kill_Chain_Stages'].apply(count_chain_stages)
print("kill chain count - min/max/mean:",
      df['kill_chain_count'].min(),
      df['kill_chain_count'].max(),
      round(df['kill_chain_count'].mean(), 2))

# attack category - one hot
attack_cat = pd.get_dummies(df['Attack_Category'], prefix='cat').astype(int)

# os - one hot
os_cat = pd.get_dummies(df['OS'], prefix='os').astype(int)

# learning objectives - multi hot
def split_objs(s):
    if pd.isna(s): return []
    return [x.strip() for x in str(s).split(';') if x.strip()]

obj_lists = df['Learning_Objectives'].apply(split_objs)
mlb = MultiLabelBinarizer()
obj_matrix = mlb.fit_transform(obj_lists)
obj_df = pd.DataFrame(obj_matrix, columns=['obj_' + c for c in mlb.classes_])

# final machine feature matrix
machine_features = pd.concat([
    df[['diff_num', 'Estimated_Time_Hours', 'Attack_Path_Length', 'kill_chain_count']].reset_index(drop=True),
    attack_cat.reset_index(drop=True),
    os_cat.reset_index(drop=True),
    obj_df.reset_index(drop=True)
], axis=1)

print("machine feature matrix shape:", machine_features.shape)
print("total objective columns:", len([c for c in machine_features.columns if c.startswith('obj_')]))

# keep clean machine info for displaying to user later
machine_info = df[['Machine_ID', 'Machine_Name', 'Platform', 'OS', 'Difficulty',
                   'Attack_Category', 'Vulnerability_Type', 'Entry_Point',
                   'Attack_Path_Length', 'Estimated_Time', 'Estimated_Time_Hours',
                   'Skills_Required', 'Learning_Objectives', 'Kill_Chain_Stages']].copy()

# save
joblib.dump(machine_features, os.path.join(MODELS_DIR_V2, 'machine_features.pkl'))
joblib.dump(machine_info, os.path.join(MODELS_DIR_V2, 'machine_info_v2.pkl'))
joblib.dump(mlb, os.path.join(MODELS_DIR_V2, 'obj_encoder.pkl'))

print("saved.")

kill chain count - min/max/mean: 3 7 4.99
machine feature matrix shape: (305, 111)
total objective columns: 99
saved.


In [37]:
# quick checks on the dataset before we move on

print("any duplicate machine names?", df['Machine_Name'].duplicated().sum())

# numeric ranges
print("\nestimated_time_hours range:", df['Estimated_Time_Hours'].min(), "-", df['Estimated_Time_Hours'].max())
print("attack_path_length range:", df['Attack_Path_Length'].min(), "-", df['Attack_Path_Length'].max())
print("kill_chain_count range:", df['kill_chain_count'].min(), "-", df['kill_chain_count'].max())

# any machines with no learning objectives?
no_obj = df['Learning_Objectives'].isna().sum() + (df['Learning_Objectives'].fillna('').str.strip() == '').sum()
print("\nmachines with no learning objectives:", no_obj)

# objective count per machine
obj_per_machine = df['Learning_Objectives'].fillna('').apply(lambda s: len([x for x in s.split(';') if x.strip()]))
print("objectives per machine - min/max/mean:", obj_per_machine.min(), "/", obj_per_machine.max(), "/", round(obj_per_machine.mean(), 2))

# the machine_features matrix - any all-zero rows?
all_zero_rows = (machine_features.sum(axis=1) == 0).sum()
print("\nall-zero machine rows:", all_zero_rows)

# any feature columns that are all zeros (useless features)?
all_zero_cols = (machine_features.sum(axis=0) == 0).sum()
print("all-zero feature columns:", all_zero_cols)

any duplicate machine names? 0

estimated_time_hours range: 0.5 - 4.5
attack_path_length range: 3 - 13
kill_chain_count range: 3 - 7

machines with no learning objectives: 0
objectives per machine - min/max/mean: 3 / 13 / 7.39

all-zero machine rows: 0
all-zero feature columns: 0


In [38]:
# building the student feature vector now, has to match machine columns exactly so the model can compare them

feature_columns = list(machine_features.columns)
print("total feature columns:", len(feature_columns))

# mapping the time bin to actual hours (midpoint of each range)
time_to_hours = {
    '<1 hour':    0.5,
    '1-2 hours':  1.5,
    '2-3 hours':  2.5,
    '3-4 hours':  3.5,
    '4+ hours':   4.5
}

# skill level is a new field for v2, will be used during training data generation
skill_to_num = {
    'Beginner':     1,
    'Intermediate': 2,
    'Advanced':     3
}

def build_student_vector(difficulty, attack_categories, os_pref,
                         learning_objectives=None, estimated_time=None,
                         skill_level=None):
    # turning the student form input into a row that matches the machine feature columns
    
    vec = pd.Series(0.0, index=feature_columns)
    
    # difficulty goes in directly
    vec['diff_num'] = diff_map.get(difficulty, 2)
    
    # estimated time defaults to 2.5h if not given
    vec['Estimated_Time_Hours'] = time_to_hours.get(estimated_time, 2.5)
    
    # student doesnt input attack path length or kill chain count so picking sensible defaults based on difficulty
    path_default = {'Easy': 4, 'Medium': 6, 'Hard': 9}
    chain_default = {'Easy': 3, 'Medium': 5, 'Hard': 7}
    vec['Attack_Path_Length'] = path_default.get(difficulty, 6)
    vec['kill_chain_count'] = chain_default.get(difficulty, 5)
    
    # attack category can be 1 or 2 values
    if isinstance(attack_categories, str):
        attack_categories = [attack_categories]
    for cat in attack_categories:
        col = 'cat_' + cat
        if col in vec.index:
            vec[col] = 1
    
    # os is single select
    os_col = 'os_' + os_pref
    if os_col in vec.index:
        vec[os_col] = 1
    
    # learning objectives are optional, 0 to 3 of them
    if learning_objectives:
        for obj in learning_objectives:
            col = 'obj_' + obj
            if col in vec.index:
                vec[col] = 1
    
    return vec


# testing with a sample student to make sure the function works
test_student = build_student_vector(
    difficulty='Easy',
    attack_categories=['Web Exploitation'],
    os_pref='Linux',
    learning_objectives=['SQL Injection', 'Credential Extraction'],
    estimated_time='1-2 hours',
    skill_level='Beginner'
)

print("\ntest student vector built ok, shape:", test_student.shape)
print("\nnon-zero values in this student:")
print(test_student[test_student != 0])

total feature columns: 111

test student vector built ok, shape: (111,)

non-zero values in this student:
diff_num                     1.0
Estimated_Time_Hours         1.5
Attack_Path_Length           4.0
kill_chain_count             3.0
cat_Web Exploitation         1.0
os_Linux                     1.0
obj_Credential Extraction    1.0
obj_SQL Injection            1.0
dtype: float64
